# Ejercicio 3: Inspección Visual de un Producto

**Curso:** IS-484 Inteligencia Artificial I  
**Práctica Calificada 001**  
**Alumno:** Aguilar Flores, Crisólogo  
**Ciclo:** 2026-II

---
## Ficha PEAS

| Componente | Descripción |
|---|---|
| **Percepción (S)** | Matriz NumPy 5×5 de píxeles: `0` = normal, `1` = defectuoso (capturada por cámara de línea de producción) |
| **Acciones (A)** | `aprobar` (producto pasa al siguiente proceso), `revision_manual` (producto desviado para inspección humana), `rechazar` (producto descartado o enviado a reproceso) |
| **Entorno (E)** | Línea de producción industrial. Totalmente observable (imagen completa disponible), estático (el producto no cambia durante la inspección), episódico (cada producto se evalúa de forma independiente), discreto |
| **Objetivo** | Maximizar la calidad del producto final descartando piezas defectuosas, minimizando el rechazo de piezas buenas y el costo de revisión manual innecesaria |
| **Medida de desempeño (P)** | Tasa de detección de defectos (recall), tasa de falsos rechazos, throughput de la línea (productos inspeccionados por minuto), costo de reproceso |

---
## Reglas de Decisión

La imagen 5×5 tiene 25 píxeles en total. Las reglas se definen así:

| Píxeles defectuosos | Acción |
|---|---|
| 0 | `aprobar` |
| 1 – 3 | `revision_manual` |
| 4 o más | `rechazar` |

### Justificación de las reglas

En una imagen de 25 píxeles que representa una pieza industrial, un producto sin ningún píxel defectuoso cumple los estándares de calidad y debe continuar en la línea sin intervención. De 1 a 3 defectos (hasta el 12% de la superficie) puede deberse a variaciones menores de iluminación, imperfecciones superficiales aceptables o errores del sensor; en estos casos, la revisión humana es más eficiente que el rechazo automático, ya que descarta ambas fuentes de error. Con 4 o más píxeles defectuosos (≥16% de la superficie) la probabilidad de que el defecto sea real y significativo es alta, por lo que el rechazo directo protege la calidad del lote y reduce el costo de inspección manual en casos claramente fuera de tolerancia. Este esquema equilibra sensibilidad de detección con eficiencia operativa.

---
## Código

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


def agente_inspeccion(imagen):
    """
    Percepcion (S): matriz 5x5 de NumPy (0 = normal, 1 = defectuoso)
    
    Parámetros:
        imagen (np.ndarray): Matriz 5×5 con valores 0 o 1.
    
    Retorna:
        tuple: (accion: str, motivo: str)
    """
    defectos = int(np.sum(imagen == 1))   # contar cuántos valores de la matriz son 1
    total    = imagen.size
    porcentaje = defectos / total * 100

    if defectos == 0:
        return ("aprobar",
                f"0 píxeles defectuosos de {total}: producto sin defectos detectados")

    elif defectos <= 3:
        return ("revision_manual",
                f"{defectos} píxel(es) defectuoso(s) de {total} ({porcentaje:.0f}%): "
                f"nivel de defectos bajo, requiere revisión humana")

    else:
        return ("rechazar",
                f"se detectaron {defectos} píxeles defectuosos de {total} ({porcentaje:.0f}%): "
                f"supera el umbral máximo permitido")


print("Función agente_inspeccion definida correctamente.")

---
## Simulación y Pruebas

Se prueban 4 matrices propias + la imagen de ejemplo del enunciado.

In [ ]:
# ---- Imagen del enunciado ----
imagen_prueba = np.array([
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
])

# ---- Matrices propias ----
# Caso A: 0 defectos → APROBAR
imagen_a = np.array([
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
])

# Caso B: 1 defecto → REVISIÓN MANUAL
imagen_b = np.array([
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
])

# Caso C: 3 defectos (borde superior de la zona de revisión) → REVISIÓN MANUAL
imagen_c = np.array([
    [1, 0, 0, 0, 1],
    [0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
])

# Caso D: 4 defectos (borde inferior, primer caso de RECHAZO) → RECHAZAR
imagen_d = np.array([
    [1, 0, 0, 0, 1],
    [0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
])

# Caso E: 10 defectos (daño severo) → RECHAZAR
imagen_e = np.array([
    [1, 1, 0, 1, 1],
    [0, 1, 0, 1, 0],
    [1, 0, 0, 0, 1],
    [0, 0, 0, 1, 0],
    [1, 0, 0, 0, 0],
])

imagenes = [
    (imagen_prueba, "Imagen del enunciado (3 defectos)"),
    (imagen_a,      "Caso A — 0 defectos       → APROBAR"),
    (imagen_b,      "Caso B — 1 defecto        → REVISIÓN MANUAL"),
    (imagen_c,      "Caso C — 3 defectos       → REVISIÓN MANUAL (borde)"),
    (imagen_d,      "Caso D — 4 defectos       → RECHAZAR (borde)"),
    (imagen_e,      "Caso E — 10 defectos      → RECHAZAR"),
]

print("=" * 68)
print("SIMULACIÓN DE INSPECCIÓN VISUAL DE PRODUCTOS")
print("=" * 68)

for imagen, desc in imagenes:
    accion, motivo = agente_inspeccion(imagen)
    print(f"\n{desc}")
    print(f"  → Acción : {accion.upper()}")
    print(f"  → Motivo : {motivo}")

print("\n" + "=" * 68)

---
## Visualización

Cada imagen se visualiza con `plt.imshow` con mapa de color invertido (blanco = normal, negro = defectuoso).

In [ ]:
# Colores de borde según decisión
border_color = {
    "aprobar":          "#2ecc71",
    "revision_manual":  "#f39c12",
    "rechazar":         "#e74c3c",
}

fig, axes = plt.subplots(2, 3, figsize=(11, 7))
axes = axes.flatten()

for ax, (imagen, desc) in zip(axes, imagenes):
    accion, motivo = agente_inspeccion(imagen)
    defectos = int(np.sum(imagen == 1))
    
    ax.imshow(imagen, cmap="gray_r", vmin=0, vmax=1, interpolation="nearest")
    
    # Cuadrícula de píxeles
    for x in range(6):
        ax.axvline(x - 0.5, color="#bdc3c7", linewidth=0.8)
    for y in range(6):
        ax.axhline(y - 0.5, color="#bdc3c7", linewidth=0.8)
    
    # Marco de color según decisión
    for spine in ax.spines.values():
        spine.set_edgecolor(border_color[accion])
        spine.set_linewidth(4)
    
    ax.set_title(f"{defectos} defecto(s) → {accion.upper()}",
                 fontsize=9, color=border_color[accion], fontweight="bold", pad=6)
    ax.set_xticks(range(5))
    ax.set_yticks(range(5))
    ax.tick_params(labelsize=7)

# Leyenda general
patches = [
    mpatches.Patch(color="#2ecc71", label="Aprobar"),
    mpatches.Patch(color="#f39c12", label="Revisión manual"),
    mpatches.Patch(color="#e74c3c", label="Rechazar"),
]
fig.legend(handles=patches, loc="lower center", ncol=3,
           framealpha=0.9, fontsize=10, title="Decisión del agente")

fig.suptitle("Agente de Inspección Visual — Matrices 5×5",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout(rect=[0, 0.07, 1, 1])
plt.savefig("inspeccion_visual.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nNota: píxeles en negro = defectuosos (valor 1), en blanco = normales (valor 0).")